# CS F425: Deep Learning Course Project (2026–27)
## Task-II: Additional Task 1 — Linear-Scale Change Magnitude & K-Means Flood Mask Generation

**Team 26**  
- **Aarnav Piparsania** (2024ADPS0729P)  
- **Samarth Soni** (2024A2PS1243P)  
- **Bhavya Gupta** (2023A7PS0494P)  

**District:** Chhatarpur, Madhya Pradesh  
**Flood Event:** 29 August – 3 September 2026 (Ken and Dhasan River Basins, Lahchura and Ban Sujara Dam Releases)  
**Deliverable Output:** `Mask_linear.tif`

---
### 1. Theoretical Background & Objectives
In Synthetic Aperture Radar (SAR) processing, calibrated backscatter $\gamma^0$ is natively represented in linear power units ($	ext{m}^2/\text{m}^2$). 
While decibel (dB) scaling applies a logarithmic transformation ($10 \log_{10}(\gamma^0)$) that compresses high-backscatter land targets and spreads out dark water features, **linear-scale change detection** measures the raw, uncompressed difference in scattered power:

$$\Delta \gamma^0_{\text{linear}} = \left| \gamma^0_{\text{before, linear}} - \gamma^0_{\text{after, linear}} \right|$$

This notebook implements:
1. Computation of linear-scale change magnitude between collocated pre-flood (May 23, 2026) and during-flood (September 3, 2026) Gamma0 backscatter.
2. Unsupervised 3-cluster MiniBatch K-Means clustering on the linear change magnitude.
3. Selection of the candidate flood cluster (highest change magnitude).
4. Binary mask generation and morphological refinement (speckle filtering, isolated blob removal, hole filling) to produce `Mask_linear.tif`.
5. Quantitative comparison and discussion of linear-scale vs. dB-scale flood mapping.

In [ ]:
# Step 1: Import Dependencies
import os
import json
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from scipy import ndimage as ndi
from skimage.morphology import disk, remove_small_objects, remove_small_holes

print("All geospatial and machine learning libraries loaded successfully.")

In [ ]:
# Step 2: Define File Paths for Chhatarpur District
base_dir = os.path.abspath("..")
proc_dir = os.path.join(base_dir, "Processed data", "Chhatarpur")
mask_dir = os.path.join(proc_dir, "Mask", "Task images")
os.makedirs(mask_dir, exist_ok=True)

before_vh_path = os.path.join(proc_dir, "Before", "VH", "Gamma0_VH.tif")
after_vh_path = os.path.join(proc_dir, "After", "VH", "Gamma0_VH.tif")
output_linear_mask = os.path.join(mask_dir, "Mask_linear.tif")

print(f"Before VH: {before_vh_path}")
print(f"After VH:  {after_vh_path}")
print(f"Target:    {output_linear_mask}")

In [ ]:
# Step 3: Compute Linear-Scale Change Magnitude
print("Loading collocated linear Gamma0 backscatter...")
# Load Before and After linear arrays
gamma_before = tifffile.imread(before_vh_path)
gamma_after = tifffile.imread(after_vh_path)

# Compute absolute difference in linear domain
valid_mask = np.isfinite(gamma_before) & np.isfinite(gamma_after) & (gamma_before > 0) & (gamma_after > 0)
linear_change = np.zeros_like(gamma_before, dtype=np.float32)
linear_change[valid_mask] = np.abs(gamma_before[valid_mask] - gamma_after[valid_mask])

print("Linear Change Statistics:")
print(f"  Min:  {linear_change[valid_mask].min():.6f}")
print(f"  Max:  {linear_change[valid_mask].max():.6f}")
print(f"  Mean: {linear_change[valid_mask].mean():.6f}")
print(f"  Std:  {linear_change[valid_mask].std():.6f}")

In [ ]:
# Step 4: MiniBatch K-Means Clustering on Linear-Scale Change Magnitude
print("Sampling valid pixels for K-Means...")
valid_pixels = linear_change[valid_mask]
# Sample 200,000 pixels for fast, robust convergence
sample_size = min(200000, len(valid_pixels))
sample_idx = np.random.choice(len(valid_pixels), size=sample_size, replace=False)
X_sample = valid_pixels[sample_idx].reshape(-1, 1)

kmeans = MiniBatchKMeans(n_clusters=3, batch_size=2048, random_state=42, n_init=3)
kmeans.fit(X_sample)

centers = kmeans.cluster_centers_.flatten()
sorted_clusters = np.argsort(centers)
flood_cluster = sorted_clusters[-1]  # Highest change magnitude cluster

print(f"Cluster Centers (Linear Scale): {centers}")
print(f"Flood candidate cluster index: {flood_cluster} (Center = {centers[flood_cluster]:.6f})")

# Predict cluster labels across the entire scene
kmeans_map = np.full(linear_change.shape, 255, dtype=np.uint8)
kmeans_map[valid_mask] = kmeans.predict(valid_pixels.reshape(-1, 1)).astype(np.uint8)

# Extract binary flood mask
raw_flood_mask = np.zeros_like(kmeans_map, dtype=np.uint8)
raw_flood_mask[valid_mask] = (kmeans_map[valid_mask] == flood_cluster).astype(np.uint8)

In [ ]:
# Step 5: Morphological Refinement
print("Applying morphological cleaning...")
# 1. Opening to remove small radar speckle
opened = ndi.binary_opening(raw_flood_mask == 1, structure=disk(1))
# 2. Remove isolated small noise blobs (< 20 connected pixels)
cleaned = remove_small_objects(opened, min_size=20)
# 3. Fill small internal holes in contiguous water bodies (< 50 pixels)
cleaned = remove_small_holes(cleaned, area_threshold=50)

mask_linear = (cleaned.astype(np.uint8)) * 255

# Save Mask_linear.tif deliverable
tifffile.imwrite(output_linear_mask, mask_linear, compression='zlib')
print(f"Successfully exported {output_linear_mask}!")

flooded_pixels = np.count_nonzero(mask_linear == 255)
pixel_area_ha = (10.0 * 10.0) / 10000.0  # 10m grid = 0.01 hectares
flooded_area_ha = flooded_pixels * pixel_area_ha
flooded_area_km2 = flooded_area_ha / 100.0
print(f"Linear-scale Flooded Pixels: {flooded_pixels:,}")
print(f"Flooded Area: {flooded_area_ha:,.2f} ha ({flooded_area_km2:,.2f} km²)")

In [ ]:
# Step 6: Visualizations & Diagnostic Comparisons
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Subsample for plotting display
stride = 4
axes[0].imshow(linear_change[::stride, ::stride], cmap='magma', vmin=0, vmax=np.percentile(linear_change[valid_mask], 98))
axes[0].set_title("Linear Change Magnitude |Before - After|")
axes[0].axis("off")

axes[1].imshow(kmeans_map[::stride, ::stride], cmap='viridis')
axes[1].set_title("3-Cluster K-Means Classification")
axes[1].axis("off")

axes[2].imshow(mask_linear[::stride, ::stride], cmap='gray')
axes[2].set_title("Refined Linear Flood Mask (Mask_linear.tif)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Discussion & Findings for Task 1
- **Dynamic Range & Skewness:** In the linear domain, backscatter values are heavily right-skewed. While land features span a broad range of linear values, low-backscatter water targets are compressed near zero.
- **K-Means Sensitivity:** When clustering in the linear domain, the cluster centroids are influenced heavily by bright scattering changes (such as agricultural harvesting, plowing, or moisture increase in non-flooded bare soil), potentially leading to false alarms in high-backscatter zones while under-detecting boundary floodwater.
- **Comparison to Decibel Scale:** The decibel scale ($10 \log_{10}(\gamma^0)$) spreads out negative backscatter values, providing superior radiometric contrast between flooded water bodies (-22 to -28 dB) and unflooded terrain (-12 to -16 dB).